In [ ]:
# ========== Colab one-cell: HYBRID HPO (PSO+GWO+WOA+GA+HHO) with SUBSET SEARCH + FULL-DATA FINAL EVAL ==========
# EDIT THESE:
ZIP_PATH = "/content/drive/MyDrive/n.zip"   # OR set DATA_DIR below
DATA_DIR = None                                             # path to unzipped folder (optional)
OUTPUT_DIR = "/content/drive/MyDrive/Enhancement_HPO_sample_then_full"
POP_SIZE = 12            # population size for hybrid HPO (reduce to speed up)
MAX_ITER = 8             # hybrid HPO generations
SAMPLE_SIZE = 3000       # sample size used during main search (keeps search fast). Set to 0 to use full dataset for search.
TOP_K_FINAL = 3          # top-K candidates to re-evaluate on full dataset
NUM_WORKERS = 6          # number of processes for parallel image evaluation
SEED = 42
# =========================================================================================

# Mount drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    pass

# Install minimal packages if missing
import importlib, os, sys, time, zipfile, random, json
for pkg in ("opencv-python-headless", "scikit-image", "Pillow", "tqdm", "numpy"):
    name = pkg.split("-")[0]
    if importlib.util.find_spec(name) is None:
        print(f"Installing {pkg} ...")
        os.system(f"pip install -q {pkg}")

# Imports
import numpy as np
import cv2
from PIL import Image, ImageEnhance
from skimage.filters import sobel
from skimage.measure import shannon_entropy
from pathlib import Path
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm

random.seed(SEED); np.random.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- Utilities: unzip / collect images -------------------------
def unzip_if_needed(zip_path, extract_to="/content/unzipped_images"):
    extract_to = Path(extract_to)
    if not extract_to.exists():
        extract_to.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {zip_path} -> {extract_to} ... (may take time)")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(str(extract_to))
    return extract_to

def collect_image_paths(root, max_images=None, shuffle=True):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
    root = Path(root)
    files = [str(p) for p in root.rglob("*") if p.suffix.lower() in exts]
    if shuffle:
        random.Random(SEED).shuffle(files)
    files.sort() if (not shuffle) else None
    if max_images and max_images > 0:
        return files[:max_images]
    return files

# --------------------- Enhancement primitives ---------------------
def apply_clahe_bgr(img, clip=2.0, tile=(8,8)):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=tile)
    L2 = clahe.apply(L)
    lab2 = cv2.merge((L2, A, B))
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)

def color_jitter_bgr(img, brightness=1.0, contrast=1.0, saturation=1.0, hue=0.0):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    if abs(brightness - 1.0) > 1e-6:
        pil = ImageEnhance.Brightness(pil).enhance(float(brightness))
    if abs(contrast - 1.0) > 1e-6:
        pil = ImageEnhance.Contrast(pil).enhance(float(contrast))
    if abs(saturation - 1.0) > 1e-6:
        pil = ImageEnhance.Color(pil).enhance(float(saturation))
    arr = np.array(pil)
    if abs(hue) > 1e-6:
        hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[...,0] = (hsv[...,0] + float(hue) * 180.0) % 180.0
        arr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def gaussian_blur(img, sigma):
    sigma = float(sigma)
    if sigma <= 0:
        return img
    k = max(3, int(2 * round(3 * sigma) + 1))
    if k % 2 == 0:
        k += 1
    return cv2.GaussianBlur(img, (k, k), sigmaX=sigma, sigmaY=sigma)

def unsharp_mask(img, amount=0.5):
    amount = float(amount)
    if amount <= 0:
        return img
    blurred = cv2.GaussianBlur(img, (0,0), sigmaX=1.0)
    return cv2.addWeighted(img, 1.0 + amount, blurred, -amount, 0)

def add_gaussian_noise(img, std):
    std = float(std)
    if std <= 0:
        return img
    noise = np.random.normal(0, std * 255.0, img.shape).astype(np.float32)
    out = img.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)

# --------------------- No-reference metrics ---------------------
def image_metrics_gray(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ent = float(shannon_entropy(gray))
    cont = float(np.std(gray))
    edge = float(np.mean(np.abs(sobel(gray.astype(np.float32) / 255.0))))
    return ent, cont, edge

# --------------------- Metaheuristic updates (PSO, GWO, WOA, GA, HHO) ---------------------
def pso_update(pop, vel, pbest, gbest, w=0.7, c1=1.5, c2=1.5):
    r1, r2 = np.random.rand(*pop.shape), np.random.rand(*pop.shape)
    vel_new = (w * vel) + c1 * r1 * (pbest - pop) + c2 * r2 * (gbest - pop)
    pop_new = pop + vel_new
    return pop_new, vel_new

def gwo_update(pop, alpha, beta, delta, a):
    N, D = pop.shape
    new_pop = np.zeros_like(pop)
    for i in range(N):
        for j in range(D):
            r1, r2 = np.random.rand(), np.random.rand()
            A1 = 2 * a * r1 - a; C1 = 2 * r2
            D_alpha = abs(C1 * alpha[j] - pop[i][j]); X1 = alpha[j] - A1 * D_alpha
            r1, r2 = np.random.rand(), np.random.rand()
            A2 = 2 * a * r1 - a; C2 = 2 * r2
            D_beta = abs(C2 * beta[j] - pop[i][j]); X2 = beta[j] - A2 * D_beta
            r1, r2 = np.random.rand(), np.random.rand()
            A3 = 2 * a * r1 - a; C3 = 2 * r2
            D_delta = abs(C3 * delta[j] - pop[i][j]); X3 = delta[j] - A3 * D_delta
            new_pop[i][j] = (X1 + X2 + X3) / 3
    return new_pop

def woa_update(pop, best, a, b=1.0):
    N, D = pop.shape
    new_pop = np.zeros_like(pop)
    for i in range(N):
        r1, r2 = np.random.rand(), np.random.rand()
        A = 2 * a * r1 - a; C = 2 * r2
        p = np.random.rand(); l = np.random.uniform(-1, 1)
        if p < 0.5:
            if abs(A) < 1:
                Dv = abs(C * best - pop[i]); new_pop[i] = best - A * Dv
            else:
                rand_idx = np.random.randint(0, N); X_rand = pop[rand_idx]
                Dv = abs(C * X_rand - pop[i]); new_pop[i] = X_rand - A * Dv
        else:
            Dv = abs(best - pop[i])
            new_pop[i] = Dv * np.exp(b * l) * np.cos(2 * np.pi * l) + best
    return new_pop

def ga_crossover_mutation(pop, fitness, crossover_rate=0.8, mutation_rate=0.1):
    N, D = pop.shape
    new_pop = np.copy(pop)
    sorted_idx = np.argsort(-fitness)
    parents = pop[sorted_idx[:max(2, N//2)]]
    for i in range(0, len(parents)-1, 2):
        if np.random.rand() < crossover_rate:
            cross_point = np.random.randint(1, D-1)
            child1 = np.concatenate((parents[i][:cross_point], parents[i+1][cross_point:]))
            child2 = np.concatenate((parents[i+1][:cross_point], parents[i][cross_point:]))
            new_pop[i], new_pop[i+1] = child1, child2
    for i in range(N):
        for j in range(D):
            if np.random.rand() < mutation_rate:
                new_pop[i][j] += np.random.normal(0, 0.05)
    return np.clip(new_pop, 0.0, 1.0)

def hho_update(pop, best, iteration, max_iter):
    N, D = pop.shape
    new_pop = np.copy(pop)
    E0 = 2 * np.random.rand() - 1
    E = 2 * E0 * (1 - iteration / max_iter)
    for i in range(N):
        q = np.random.rand(); r = np.random.rand()
        if abs(E) >= 1:
            rand_idx = np.random.randint(0, N); X_rand = pop[rand_idx]
            new_pop[i] = X_rand - r * abs(X_rand - 2 * r * pop[i])
        else:
            if q >= 0.5 and abs(E) < 0.5:
                new_pop[i] = best - E * abs(best - pop[i])
            elif q >= 0.5 and abs(E) >= 0.5:
                jump_strength = 2 * (1 - r)
                new_pop[i] = best - E * abs(jump_strength * best - pop[i])
            elif q < 0.5 and abs(E) >= 0.5:
                Y = best - E * abs(best - pop[i]); Z = Y + np.random.randn(D) * abs(Y - best); new_pop[i] = Z
            else:
                Y = best - E * abs(best - pop[i]); Z = Y + E * np.random.randn(D); new_pop[i] = Z
    return np.clip(new_pop, 0.0, 1.0)

# --------------------- Parameter encoding (big ranges) ---------------------
D = 11
def sample_population(N):
    return np.random.rand(N, D)

def decode_vector(v):
    return {
        "clahe_prob": float(v[0]),                         # 0..1 blend factor
        "clahe_clip": float(1.0 + v[1] * 7.0),             # 1.0..8.0
        "clahe_tile": (8,8) if v[2] < 0.33 else ((16,16) if v[2]<0.66 else (32,32)),
        "brightness": float(0.5 + v[3] * 1.2),             # 0.5..1.7
        "contrast": float(0.5 + v[4] * 1.2),
        "saturation": float(0.5 + v[5] * 1.2),
        "hue": float(-0.2 + v[6] * 0.4),                   # -0.2..0.2
        "blur_sigma": float(v[7] * 3.0),                   # 0..3.0
        "sharpen": float(v[8] * 2.0),                      # 0..2.0
        "noise_std": float(v[9] * 0.05),                   # 0..0.05
        "unsharp_prob": float(v[10])                       # 0..1
    }

# --------------------- Deterministic enhancement pipeline ---------------------
def enhance_image_deterministic(img, params):
    clahe_img = apply_clahe_bgr(img, clip=params["clahe_clip"], tile=params["clahe_tile"])
    out = ((1.0 - params["clahe_prob"]) * img.astype(np.float32) + params["clahe_prob"] * clahe_img.astype(np.float32)).round().astype(np.uint8)
    jittered = color_jitter_bgr(out, brightness=params["brightness"], contrast=params["contrast"], saturation=params["saturation"], hue=params["hue"])
    out = (0.6 * out.astype(np.float32) + 0.4 * jittered.astype(np.float32)).round().astype(np.uint8)
    blurred = gaussian_blur(out, params["blur_sigma"])
    sharpened = unsharp_mask(blurred, amount=params["sharpen"]) if params["sharpen"] > 1e-6 else blurred
    out = ((1.0 - params["unsharp_prob"]) * blurred.astype(np.float32) + params["unsharp_prob"] * sharpened.astype(np.float32)).round().astype(np.uint8)
    if params["noise_std"] > 1e-9:
        out = add_gaussian_noise(out, params["noise_std"])
    return np.clip(out, 0, 255).astype(np.uint8)

# --------------------- Per-image evaluator (deterministic) ---------------------
def eval_single_image(path, params):
    try:
        img = cv2.imread(path)
        if img is None:
            return None
    except Exception:
        return None
    try:
        orig = image_metrics_gray(img)
        enh = image_metrics_gray(enhance_image_deterministic(img, params))
        return (orig, enh)
    except Exception:
        return None

# --------------------- Evaluate candidate across images (parallel) ---------------------
def evaluate_candidate_on_images(vec, image_paths, num_workers=NUM_WORKERS):
    params = decode_vector(vec)
    func = partial(eval_single_image, params=params)
    results = []
    with Pool(processes=num_workers) as pool:
        for r in pool.imap(func, image_paths, chunksize=32):
            if r is None:
                continue
            results.append(r)
    if len(results) == 0:
        return -np.inf, params
    origs = np.array([r[0] for r in results], dtype=float)
    enhs  = np.array([r[1] for r in results], dtype=float)
    eps = 1e-9
    rel_gain = (enhs - origs) / (origs + eps)
    weights = np.array([0.45, 0.30, 0.25])
    per_image_score = rel_gain.dot(weights)
    fitness = float(np.mean(per_image_score))
    return fitness, params

# --------------------- Hybrid search (subset) then final eval (full dataset) ---------------------
def hybrid_hpo_with_sampling(image_paths_for_search, image_paths_full, pop_size=POP_SIZE, max_iter=MAX_ITER):
    np.random.seed(SEED); random.seed(SEED)
    pop = sample_population(pop_size)
    vel = np.zeros_like(pop)
    fitness = np.full(pop_size, -np.inf, dtype=float)
    decoded = [None] * pop_size
    pbest = pop.copy()
    pbest_fit = np.full(pop_size, -np.inf, dtype=float)
    gbest = pop[0].copy()
    gbest_fit = -np.inf

    # Evaluate initial population on sampled subset
    print("Evaluating initial population on sampled subset...")
    for i in range(pop_size):
        fit, params = evaluate_candidate_on_images(pop[i], image_paths_for_search, num_workers=NUM_WORKERS)
        fitness[i] = fit
        decoded[i] = params
        pbest_fit[i] = fit; pbest[i] = pop[i].copy()
        if fit > gbest_fit:
            gbest_fit = fit; gbest = pop[i].copy()
        print(f" init ind {i}: fitness={fit:.6f}")

    # Hybrid loop (PSO->GWO->WOA->GA->HHO) each generation, evaluated on sampled subset
    for it in range(1, max_iter + 1):
        print(f"\n=== ITER {it}/{max_iter} (subset evaluation) — best_so_far={gbest_fit:.6f} ===")
        idx_sorted = np.argsort(-fitness)
        alpha = pop[idx_sorted[0]].copy()
        beta  = pop[idx_sorted[1] if len(idx_sorted)>1 else idx_sorted[0]].copy()
        delta = pop[idx_sorted[2] if len(idx_sorted)>2 else idx_sorted[0]].copy()
        a = 2 * (1 - it / max_iter)

        pop_pso, vel = pso_update(pop, vel, pbest, gbest.reshape(1, -1), w=0.7, c1=1.5, c2=1.5)
        pop_gwo = gwo_update(pop_pso, alpha, beta, delta, a)
        pop_woa = woa_update(pop_gwo, gbest, a)
        pop_ga  = ga_crossover_mutation(pop_woa, fitness, crossover_rate=0.75, mutation_rate=0.08)
        pop_hho = hho_update(pop_ga, gbest, it, max_iter)
        pop = np.clip(pop_hho, 0.0, 1.0)

        # evaluate new population on sampled subset
        for i in range(pop_size):
            fit, params = evaluate_candidate_on_images(pop[i], image_paths_for_search, num_workers=NUM_WORKERS)
            fitness[i] = fit
            decoded[i] = params
            if fit > pbest_fit[i]:
                pbest_fit[i] = fit; pbest[i] = pop[i].copy()
            if fit > gbest_fit:
                gbest_fit = fit; gbest = pop[i].copy()
            print(f" it{it} ind{i}: fitness={fit:.6f} (pbest={pbest_fit[i]:.6f})")

    # After search on subset: get top-K candidates by fitness (on subset)
    top_k_idx = np.argsort(-fitness)[:TOP_K_FINAL]
    top_candidates = [(int(idx), float(fitness[idx]), decode_vector(pop[idx])) for idx in top_k_idx]

    # Final evaluation: evaluate top-K on FULL dataset (more reliable)
    print("\nEvaluating top-K candidates on FULL dataset for final ranking...")
    final_results = []
    for idx, fit_subset, cand in top_candidates:
        print(f" Re-evaluating candidate {idx} on full dataset...")
        fit_full, params = evaluate_candidate_on_images(pop[idx], image_paths_full, num_workers=NUM_WORKERS)
        final_results.append({"index": int(idx), "fitness_subset": float(fit_subset), "fitness_full": float(fit_full), "params": params})
        print(f"  -> fitness_full={fit_full:.6f}")

    # choose best by full-dataset fitness (if all -inf fallback to subset)
    best_full_idx = max(range(len(final_results)), key=lambda i: final_results[i]["fitness_full"])
    best_candidate = final_results[best_full_idx]

    return {"best_from_full": best_candidate, "topk_full": final_results, "population_subset_fitness": fitness, "pop_vectors": pop}

# --------------------- Prepare image lists and run ---------------------
start_time = time.time()

# prepare image root
if DATA_DIR:
    image_root = Path(DATA_DIR)
else:
    if ZIP_PATH is None:
        raise ValueError("Set ZIP_PATH or DATA_DIR")
    image_root = unzip_if_needed(ZIP_PATH, extract_to="/content/unzipped_images")

# sample for search vs full evaluation
if SAMPLE_SIZE and SAMPLE_SIZE > 0:
    image_paths_for_search = collect_image_paths(image_root, max_images=min(SAMPLE_SIZE, 100000), shuffle=True)
else:
    image_paths_for_search = collect_image_paths(image_root, max_images=None, shuffle=True)

# full list for final eval (use all images)
image_paths_full = collect_image_paths(image_root, max_images=None, shuffle=False)

print(f"Images: search subset = {len(image_paths_for_search)}, full dataset = {len(image_paths_full)}")

# run hybrid HPO with sampling
results = hybrid_hpo_with_sampling(image_paths_for_search, image_paths_full, pop_size=POP_SIZE, max_iter=MAX_ITER)

elapsed = (time.time() - start_time) / 60.0
print("\n=== DONE ===")
print(f"Elapsed time: {elapsed:.2f} minutes")

# Save results (best + top-K full results)
out_path = os.path.join(OUTPUT_DIR, "hpo_sample_then_full_results.json")
with open(out_path, "w") as f:
    json.dump({"seed": SEED, "config": {"POP_SIZE": POP_SIZE, "MAX_ITER": MAX_ITER, "SAMPLE_SIZE": SAMPLE_SIZE, "TOP_K_FINAL": TOP_K_FINAL, "NUM_WORKERS": NUM_WORKERS}, "results": results}, f, indent=2)

print("Saved results to:", out_path)
print("Best candidate (by full-dataset eval):")
import pprint
pprint.pprint(results["best_from_full"])


Mounted at /content/drive
Installing opencv-python-headless ...
Installing scikit-image ...
Installing Pillow ...
Extracting /content/drive/MyDrive/n.zip -> /content/unzipped_images ... (may take time)
Images: search subset = 3000, full dataset = 36656
Evaluating initial population on sampled subset...
 init ind 0: fitness=0.072290
 init ind 1: fitness=0.374821
 init ind 2: fitness=0.096832
 init ind 3: fitness=0.675316
 init ind 4: fitness=0.247913
 init ind 5: fitness=-0.083513
 init ind 6: fitness=0.201447
 init ind 7: fitness=0.242911
 init ind 8: fitness=0.370682
 init ind 9: fitness=-0.053222
 init ind 10: fitness=0.347831
 init ind 11: fitness=0.509364

=== ITER 1/8 (subset evaluation) — best_so_far=0.675316 ===
 it1 ind0: fitness=0.058323 (pbest=0.072290)
 it1 ind1: fitness=0.317768 (pbest=0.374821)
 it1 ind2: fitness=0.475501 (pbest=0.475501)
 it1 ind3: fitness=0.787922 (pbest=0.787922)
 it1 ind4: fitness=0.270488 (pbest=0.270488)
 it1 ind5: fitness=0.310762 (pbest=0.310762)
 

TypeError: Object of type ndarray is not JSON serializable

In [ ]:
# ===== Resume & Save: make results JSON-serializable, save JSON + .npy artifacts =====
import json, os, sys
import numpy as np
import types
import torch

# Use existing OUTPUT_DIR if present in notebook namespace, otherwise fallback:
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = "/content/drive/MyDrive/Enhancement_HPO_sample_then_full_results_fixed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Try to find `results` in current namespace
if 'results' not in globals():
    # try alternative variable names that might exist
    alt = None
    for name in ('best_from_full', 'final_results', 'final_result', 'hpo_results', 'res'):
        if name in globals():
            alt = name
            break
    if alt is None:
        print("ERROR: No `results` variable found in current notebook state.")
        print("Available globals (first 60):", list(globals().keys())[:60])
        raise NameError("No `results` found — can't resume saving.")
    else:
        print(f"Note: `results` not found; using `{alt}` instead.")
        results = globals()[alt]
else:
    results = globals()['results']

# Recursive converter for non-JSON types
def make_json_serializable(obj):
    # numpy arrays
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    # torch tensors
    if 'torch' in sys.modules and isinstance(obj, torch.Tensor):
        return make_json_serializable(obj.cpu().detach().numpy())
    # numpy scalar types
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    # native python types that are JSONable
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    # dict -> recurse
    if isinstance(obj, dict):
        return {str(k): make_json_serializable(v) for k,v in obj.items()}
    # list / tuple / set -> list
    if isinstance(obj, (list, tuple, set)):
        return [make_json_serializable(v) for v in obj]
    # types.SimpleNamespace
    if isinstance(obj, types.SimpleNamespace):
        return make_json_serializable(vars(obj))
    # if object has __dict__, try converting that
    if hasattr(obj, "__dict__"):
        try:
            return make_json_serializable(vars(obj))
        except Exception:
            pass
    # fallback: try str()
    try:
        return str(obj)
    except Exception:
        return None

# Clean the results for JSON (but keep original in memory)
cleaned = make_json_serializable(results)

# Save JSON
json_path = os.path.join(OUTPUT_DIR, "hpo_sample_then_full_results_fixed.json")
with open(json_path, "w") as f:
    json.dump(cleaned, f, indent=2)
print(f"Saved cleaned JSON to: {json_path}")

# Additionally save large numeric arrays separately (if present in results)
# Common keys to check: 'pop_vectors', 'population_subset_fitness', 'pop'
saved_npy = []
if isinstance(results, dict):
    for candidate_key in ("pop_vectors", "population_subset_fitness", "pop_vectors_np", "population"):
        if candidate_key in results:
            val = results[candidate_key]
            try:
                arr = np.array(val)
                npy_path = os.path.join(OUTPUT_DIR, f"{candidate_key}.npy")
                np.save(npy_path, arr)
                saved_npy.append(npy_path)
            except Exception:
                pass

# If results contains 'topk_full' or 'best_from_full', save them separately in readable form
if isinstance(results, dict):
    if "topk_full" in results:
        try:
            topk_clean = make_json_serializable(results["topk_full"])
            p = os.path.join(OUTPUT_DIR, "topk_full.json")
            with open(p, "w") as f:
                json.dump(topk_clean, f, indent=2)
            print("Saved top-K full results to:", p)
        except Exception:
            pass
    if "best_from_full" in results:
        try:
            best_clean = make_json_serializable(results["best_from_full"])
            p = os.path.join(OUTPUT_DIR, "best_from_full.json")
            with open(p, "w") as f:
                json.dump(best_clean, f, indent=2)
            print("Saved best candidate to:", p)
        except Exception:
            pass

# Print a concise summary for quick resume info
print("\n=== Summary (quick) ===")
try:
    # try to extract best candidate info robustly
    if isinstance(results, dict):
        best = results.get("best_from_full") or results.get("best_candidate") or results.get("best")
        if best:
            print("Best candidate (raw):")
            print(make_json_serializable(best))
        elif "topk_full" in results:
            print("Top-K full (first entry):")
            print(make_json_serializable(results["topk_full"][:1]))
        else:
            print("Couldn't find best candidate inside results dict; dumping cleaned top-level keys:")
            print(list(results.keys()))
    else:
        print("`results` isn't a dict. Type:", type(results))
        print("Content (short):", make_json_serializable(results) if len(str(results))<1000 else str(results)[:1000])
except Exception as e:
    print("Error while summarizing:", e)

if saved_npy:
    print("\nSaved .npy artifacts:")
    for p in saved_npy:
        print(" -", p)

print("\nAll done — you can now continue further processing. The JSON is safe and readable.")


Saved cleaned JSON to: /content/drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results_fixed.json
Saved top-K full results to: /content/drive/MyDrive/Enhancement_HPO_sample_then_full/topk_full.json
Saved best candidate to: /content/drive/MyDrive/Enhancement_HPO_sample_then_full/best_from_full.json

=== Summary (quick) ===
Best candidate (raw):
{'index': 2, 'fitness_subset': 2.1100515198045153, 'fitness_full': 2.132937591983258, 'params': {'clahe_prob': 1.0, 'clahe_clip': 8.0, 'clahe_tile': [32, 32], 'brightness': 1.2759188517019069, 'contrast': 0.9493431339288161, 'saturation': 1.6998323538384155, 'hue': 0.16715379261674557, 'blur_sigma': 0.3270978297105927, 'sharpen': 2.0, 'noise_std': 0.03727316957180657, 'unsharp_prob': 1.0}}

Saved .npy artifacts:
 - /content/drive/MyDrive/Enhancement_HPO_sample_then_full/pop_vectors.npy
 - /content/drive/MyDrive/Enhancement_HPO_sample_then_full/population_subset_fitness.npy

All done — you can now continue further processing. 

In [ ]:
# Finds result files and prints the parameter sets they contain (Colab)
import json, os, glob, numpy as np, pprint

candidates = [
    "best_from_full.json",
    "topk_full.json",
    "hpo_sample_then_full_results_fixed.json",
    "hpo_sample_then_full_results.json",
    "pop_vectors.npy",
    "population_subset_fitness.npy",
    "population_subset_fitness.npy"
]

# search common locations
search_roots = ["/content", "/content/drive", "."]
found = {}

for root in search_roots:
    for name in candidates:
        for path in glob.glob(os.path.join(root, "**", name), recursive=True):
            found.setdefault(name, []).append(path)

if not found:
    print("No result files found under /content or /content/drive. Adjust search paths or upload files.")
else:
    pprint.pprint({k: v for k, v in found.items()})

def try_load_json(path):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception as e:
        print("Failed to load json", path, "->", e)
        return None

print("\n--- Parsing JSON results (if present) ---\n")
# Check best_from_full.json
if "best_from_full.json" in found:
    for p in found["best_from_full.json"]:
        print("==>", p)
        j = try_load_json(p)
        if j is None:
            continue
        # common locations in the saved structure
        # The saved file could be the best object or a wrapper with 'params'
        if isinstance(j, dict):
            pprint.pprint(j)
            # try to extract params if present
            if "params" in j:
                print("\nDecoded parameters (params):")
                pprint.pprint(j["params"])
            elif "best_from_full" in j and isinstance(j["best_from_full"], dict):
                print("\nbest_from_full -> params:")
                pprint.pprint(j["best_from_full"].get("params"))
        else:
            pprint.pprint(j)
        print()

# Check topk_full.json
if "topk_full.json" in found:
    for p in found["topk_full.json"]:
        print("==>", p)
        j = try_load_json(p)
        if j is None:
            continue
        if isinstance(j, list):
            print(f"Top-{len(j)} entries found. Showing params for each:")
            for i, entry in enumerate(j):
                print(f"--- entry {i} ---")
                pprint.pprint(entry.get("params") if isinstance(entry, dict) else entry)
        elif isinstance(j, dict):
            # might be wrapper with key 'topk_full'
            if "topk_full" in j:
                for i, entry in enumerate(j["topk_full"]):
                    print(f"--- topk_full entry {i} ---")
                    pprint.pprint(entry.get("params"))
            else:
                pprint.pprint(j)
        print()

# Check cleaned/hpo results file
if "hpo_sample_then_full_results_fixed.json" in found:
    for p in found["hpo_sample_then_full_results_fixed.json"]:
        print("==>", p)
        j = try_load_json(p)
        if j is None:
            continue
        # Try to locate best_from_full or topk_full inside
        if isinstance(j, dict):
            keys = list(j.keys())
            print("Top-level keys:", keys)
            if "results" in j and isinstance(j["results"], dict):
                res = j["results"]
                print("Results keys:", list(res.keys()))
                if "best_from_full" in res:
                    print("\nBest_from_full -> params:")
                    pprint.pprint(res["best_from_full"].get("params"))
                if "topk_full" in res:
                    print("\nTop-K params (first entries):")
                    for i, e in enumerate(res["topk_full"][:5]):
                        print(f" - rank {i}:")
                        pprint.pprint(e.get("params"))
        else:
            pprint.pprint(j)
        print()

# Show pop_vectors.npy info
if "pop_vectors.npy" in found:
    for p in found["pop_vectors.npy"]:
        print("==> pop_vectors.npy at", p)
        try:
            arr = np.load(p, allow_pickle=True)
            print("pop_vectors dtype/shape:", type(arr), getattr(arr, "shape", None))
            # if 2D, show first row decoded approx
            if hasattr(arr, "shape") and len(arr.shape) >= 2:
                print("First vector (raw):", arr[0])
        except Exception as e:
            print("Failed to load pop_vectors.npy:", e)
        print()

# If nothing printed useful, tell user next steps
print("\n--- Summary advice ---")
print("• Most human-friendly files with parameters: 'best_from_full.json', 'topk_full.json', or the cleaned 'hpo_sample_then_full_results_fixed.json'.")
print("• 'pop_vectors.npy' contains raw numeric vectors (you can decode them with the decode_vector function used in your notebook).")
print("\nIf you want, tell me which file path printed above you prefer and I will print only the parameter set in a compact form or produce a CSV of parameters.")


{'best_from_full.json': ['/content/drive/MyDrive/Enhancement_HPO_sample_then_full/best_from_full.json',
                         '/content/drive/MyDrive/Enhancement_HPO_sample_then_full/best_from_full.json',
                         './drive/MyDrive/Enhancement_HPO_sample_then_full/best_from_full.json'],
 'hpo_sample_then_full_results.json': ['/content/drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results.json',
                                       '/content/drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results.json',
                                       './drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results.json'],
 'hpo_sample_then_full_results_fixed.json': ['/content/drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results_fixed.json',
                                             '/content/drive/MyDrive/Enhancement_HPO_sample_then_full/hpo_sample_then_full_results_fixed.json',
                